# Complete Data Science Course in Python

This notebook is designed for classroom teaching. It uses one self-contained test dataset and walks through the most important stages of a data science project:

1. What data science is
2. Python setup and imports
3. Creating a test dataset
4. Exploratory Data Analysis, also called EDA
5. Data cleaning
6. Feature engineering
7. Train-test split
8. Preprocessing numeric and categorical data
9. Classification model
10. Regression model
11. Model evaluation
12. Cross-validation
13. Hyperparameter tuning
14. Clustering
15. Dimensionality reduction with PCA
16. Saving and loading a trained model
17. Student exercises

The example problem: predict whether a student will pass an exam and estimate the student's final score.

## 1. What Is Data Science?

Data science is the process of using data to answer questions, make predictions, support decisions, and discover patterns.

A typical data science workflow is:

1. Define the problem.
2. Collect or create data.
3. Clean the data.
4. Explore the data.
5. Prepare features for machine learning.
6. Train models.
7. Evaluate models.
8. Communicate results.
9. Deploy or reuse the model.

Important vocabulary:

- **Dataset**: A table of data.
- **Row / observation**: One example, such as one student.
- **Column / variable**: One measured value, such as study hours.
- **Feature**: An input used by a model.
- **Target / label**: The output we want to predict.
- **Model**: A mathematical system that learns patterns from data.
- **Training data**: Data used to teach the model.
- **Test data**: Data used to check how well the model performs on unseen examples.

## 2. Setup

If any library is missing, install it from a terminal or notebook cell with:

```python
%pip install numpy pandas matplotlib scikit-learn joblib
```

Run the imports below before continuing.

In [2]:
# Run this setup cell first.
# It checks whether the required packages are installed and installs missing ones.
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    try:
        # In Jupyter, %pip installs packages into the active notebook kernel.
        from IPython import get_ipython

        ipython = get_ipython()
        if ipython is not None:
            ipython.run_line_magic("pip", "install " + " ".join(missing_packages))
        else:
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    except Exception as error:
        print("Automatic installation did not work in this Python environment.")
        print("Please run this in a notebook cell instead:")
        print("%pip install numpy pandas matplotlib scikit-learn joblib")
        raise error
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [3]:
# Core data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn tools for machine learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import joblib

# Display settings make notebook output easier to read.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# A fixed random seed makes results reproducible for teaching.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 3. Create a Test Dataset

In real projects, data may come from CSV files, Excel, SQL databases, APIs, surveys, sensors, or web applications.

For teaching, this notebook creates a synthetic student-performance dataset. Synthetic means the data is generated by code, but it behaves like a realistic dataset.

Each row represents one student.

In [ ]:
# Number of students in our test dataset
n_students = 500

# Create input features.
# np.random.normal creates bell-shaped numeric values.
# np.random.choice creates categorical values.
study_hours = np.clip(np.random.normal(loc=5.5, scale=2.0, size=n_students), 0.5, 12)
attendance_rate = np.clip(np.random.normal(loc=78, scale=12, size=n_students), 35, 100)
previous_score = np.clip(np.random.normal(loc=62, scale=14, size=n_students), 20, 100)
sleep_hours = np.clip(np.random.normal(loc=6.8, scale=1.1, size=n_students), 3.5, 10)
internet_access = np.random.choice(["Yes", "No"], size=n_students, p=[0.82, 0.18])
parent_education = np.random.choice(
    ["High School", "Bachelor", "Master", "PhD", "Other"],
    size=n_students,
    p=[0.30, 0.36, 0.18, 0.04, 0.12],
)
extra_classes = np.random.choice(["Yes", "No"], size=n_students, p=[0.35, 0.65])

# Create a realistic final score using a formula plus random noise.
# The target depends on study hours, attendance, previous score, sleep, and categories.
noise = np.random.normal(loc=0, scale=7, size=n_students)
final_score = (
    0.35 * previous_score
    + 2.8 * study_hours
    + 0.22 * attendance_rate
    + 1.5 * sleep_hours
    + np.where(internet_access == "Yes", 3, -3)
    + np.where(extra_classes == "Yes", 4, 0)
    + np.where(parent_education == "Bachelor", 2, 0)
    + np.where(parent_education == "Master", 4, 0)
    + np.where(parent_education == "PhD", 5, 0)
    + noise
)
final_score = np.clip(final_score, 0, 100).round(1)

# Classification target: pass or fail.
# We use 70 as the passing mark so the teaching dataset has both classes clearly represented.
passed = np.where(final_score >= 70, 1, 0)

# Combine columns into one pandas DataFrame.
df = pd.DataFrame(
    {
        "student_id": range(1, n_students + 1),
        "study_hours": study_hours.round(1),
        "attendance_rate": attendance_rate.round(1),
        "previous_score": previous_score.round(1),
        "sleep_hours": sleep_hours.round(1),
        "internet_access": internet_access,
        "parent_education": parent_education,
        "extra_classes": extra_classes,
        "final_score": final_score,
        "passed": passed,
    }
)

# Add a few missing values to teach data cleaning.
missing_indices = np.random.choice(df.index, size=35, replace=False)
df.loc[missing_indices[:12], "study_hours"] = np.nan
df.loc[missing_indices[12:24], "attendance_rate"] = np.nan
df.loc[missing_indices[24:], "internet_access"] = np.nan

# Add a few duplicate rows to show duplicate handling.
df = pd.concat([df, df.sample(5, random_state=RANDOM_SEED)], ignore_index=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (505, 10)


,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
0,1,6.5,89.1,81.6,7.7,Yes,High School,Yes,82.2,1
1,2,5.2,100.0,74.9,6.2,Yes,High School,No,78.2,1
2,3,6.8,61.2,62.8,5.9,Yes,High School,No,67.6,0
3,4,8.5,84.8,52.9,6.8,NaN,Bachelor,Yes,83.3,1
4,5,NaN,70.2,71.8,6.6,Yes,High School,No,69.1,0


### Save the Test Dataset as a CSV File

This makes the dataset available outside the notebook too. Students can practice loading it later.

In [5]:
df.to_csv("student_performance_test_dataset.csv", index=False)
print("Saved: student_performance_test_dataset.csv")

Saved: student_performance_test_dataset.csv


## 4. Load Data

Most data science projects begin by loading data. Here we load the CSV file we just created.

In [6]:
data = pd.read_csv("student_performance_test_dataset.csv")
data.head()

,student_id,study_hours,attendance_rate,previous_score,sleep_hours,internet_access,parent_education,extra_classes,final_score,passed
0,1,6.5,89.1,81.6,7.7,Yes,High School,Yes,82.2,1
1,2,5.2,100.0,74.9,6.2,Yes,High School,No,78.2,1
2,3,6.8,61.2,62.8,5.9,Yes,High School,No,67.6,0
3,4,8.5,84.8,52.9,6.8,NaN,Bachelor,Yes,83.3,1
4,5,NaN,70.2,71.8,6.6,Yes,High School,No,69.1,0


## 5. Exploratory Data Analysis

EDA means exploring data before modeling. The goal is to understand the data's structure, quality, patterns, and possible problems.

In [7]:
# Shape tells us number of rows and columns.
print("Rows, columns:", data.shape)

# info() shows column names, non-null values, and data types.
data.info()

Rows, columns: (505, 10)
<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        505 non-null    int64  
 1   study_hours       493 non-null    float64
 2   attendance_rate   493 non-null    float64
 3   previous_score    505 non-null    float64
 4   sleep_hours       505 non-null    float64
 5   internet_access   493 non-null    str    
 6   parent_education  505 non-null    str    
 7   extra_classes     505 non-null    str    
 8   final_score       505 non-null    float64
 9   passed            505 non-null    int64  
dtypes: float64(5), int64(2), str(3)
memory usage: 39.6 KB


In [8]:
# describe() summarizes numeric columns.
data.describe()

,student_id,study_hours,attendance_rate,previous_score,sleep_hours,final_score,passed
count,505.000000,493.000000,493.000000,505.000000,505.000000,505.000000,505.000000
mean,250.142574,5.544016,78.259838,63.582772,6.835446,70.691881,0.526733
std,144.377054,1.955841,11.426800,14.154383,1.081221,10.503788,0.499780
min,1.000000,0.500000,45.600000,21.500000,3.600000,39.000000,0.000000
25%,125.000000,4.100000,71.100000,53.600000,6.100000,63.700000,0.000000
50%,250.000000,5.500000,78.300000,63.700000,6.800000,70.400000,1.000000
75%,375.000000,6.800000,85.800000,72.600000,7.600000,77.800000,1.000000
max,500.000000,12.000000,100.000000,98.400000,10.000000,100.000000,1.000000


In [9]:
# For categorical columns, value_counts() shows category frequency.
categorical_columns = ["internet_access", "parent_education", "extra_classes", "passed"]

for col in categorical_columns:
    print("\n", col)
    print(data[col].value_counts(dropna=False))


 internet_access
internet_access
Yes    408
No      85
NaN     12
Name: count, dtype: int64

 parent_education
parent_education
High School    214
Bachelor       184
Master          90
PhD             17
Name: count, dtype: int64

 extra_classes
extra_classes
No     324
Yes    181
Name: count, dtype: int64

 passed
passed
1    266
0    239
Name: count, dtype: int64


In [ ]:
# Check missing values in each column.
data.isna().sum()

In [ ]:
# Check duplicate rows.
print("Duplicate rows:", data.duplicated().sum())

## 6. Basic Visualization

Visualizations help us understand distributions, relationships, and unusual values.

In [ ]:
# Histogram of final scores
plt.figure(figsize=(8, 5))
plt.hist(data["final_score"], bins=20, edgecolor="black")
plt.title("Distribution of Final Scores")
plt.xlabel("Final Score")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# Scatter plot: study hours vs final score
plt.figure(figsize=(8, 5))
plt.scatter(data["study_hours"], data["final_score"], alpha=0.65)
plt.title("Study Hours vs Final Score")
plt.xlabel("Study Hours")
plt.ylabel("Final Score")
plt.show()

In [ ]:
# Correlation measures linear relationship between numeric columns.
# Values near 1 mean strong positive relation; near -1 mean strong negative relation.
numeric_data = data.select_dtypes(include=["int64", "float64"])
numeric_data.corr().round(2)

## 7. Data Cleaning

Common cleaning tasks:

- Remove duplicate rows.
- Handle missing values.
- Fix wrong data types.
- Remove impossible values.
- Standardize categories.

For machine learning, we usually place cleaning and preprocessing inside a pipeline. Still, it is useful to understand the manual steps.

In [ ]:
# Make a copy so we do not accidentally damage the original data.
clean_data = data.copy()

# Remove exact duplicate rows.
clean_data = clean_data.drop_duplicates()

print("Original shape:", data.shape)
print("After removing duplicates:", clean_data.shape)

In [ ]:
# Manual missing-value handling for demonstration.
# Numeric columns: fill with median.
# Categorical columns: fill with most common value.
clean_data["study_hours"] = clean_data["study_hours"].fillna(clean_data["study_hours"].median())
clean_data["attendance_rate"] = clean_data["attendance_rate"].fillna(clean_data["attendance_rate"].median())
clean_data["internet_access"] = clean_data["internet_access"].fillna(clean_data["internet_access"].mode()[0])

clean_data.isna().sum()

## 8. Feature Engineering

Feature engineering means creating useful input columns for a model.

Example: a student with high study hours and high attendance may have better learning behavior, so we create a combined feature.

In [ ]:
clean_data["study_attendance_index"] = clean_data["study_hours"] * (clean_data["attendance_rate"] / 100)
clean_data[["study_hours", "attendance_rate", "study_attendance_index"]].head()

## 9. Prepare Features and Targets

We will solve two machine learning problems:

1. **Classification**: Predict whether the student passed.
2. **Regression**: Predict the final score.

`student_id` is not useful for prediction because it is just an identifier.

In [ ]:
feature_columns = [
    "study_hours",
    "attendance_rate",
    "previous_score",
    "sleep_hours",
    "internet_access",
    "parent_education",
    "extra_classes",
    "study_attendance_index",
]

X = clean_data[feature_columns]
y_classification = clean_data["passed"]
y_regression = clean_data["final_score"]

X.head()

## 10. Train-Test Split

We split the data into training and testing sets.

- The model learns from the training set.
- The test set estimates how the model may perform on new students.

A common split is 80% training and 20% testing.

In [ ]:
X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X,
    y_classification,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y_classification,
)

# Use the same row indexes for regression so X and y stay perfectly aligned.
y_train_reg = y_regression.loc[X_train.index]
y_test_reg = y_regression.loc[X_test.index]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 11. Preprocessing Pipeline

Machine learning models need numeric inputs.

Preprocessing steps:

- Numeric columns: fill missing values and scale.
- Categorical columns: fill missing values and one-hot encode.

A pipeline prevents data leakage and keeps training code organized.

In [ ]:
numeric_features = ["study_hours", "attendance_rate", "previous_score", "sleep_hours", "study_attendance_index"]
categorical_features = ["internet_access", "parent_education", "extra_classes"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 12. Classification: Predict Pass or Fail

Classification predicts a category. Here the categories are:

- `1`: passed
- `0`: failed

In [ ]:
classification_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

# Train the model.
classification_model.fit(X_train, y_train_cls)

# Make predictions on unseen test data.
y_pred_cls = classification_model.predict(X_test)

print("First 10 predictions:", y_pred_cls[:10])

### Classification Metrics

- **Accuracy**: How many predictions were correct overall?
- **Precision**: Of predicted passes, how many were actually passes?
- **Recall**: Of actual passes, how many did the model find?
- **F1 score**: Balance between precision and recall.
- **Confusion matrix**: Table of correct and incorrect predictions.

In [ ]:
print("Accuracy:", round(accuracy_score(y_test_cls, y_pred_cls), 3))
print("Precision:", round(precision_score(y_test_cls, y_pred_cls), 3))
print("Recall:", round(recall_score(y_test_cls, y_pred_cls), 3))
print("F1 score:", round(f1_score(y_test_cls, y_pred_cls), 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_cls, y_pred_cls))

print("\nClassification report:")
print(classification_report(y_test_cls, y_pred_cls))

## 13. Regression: Predict Final Score

Regression predicts a number. Here we predict the student's final exam score.

In [ ]:
regression_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

regression_model.fit(X_train, y_train_reg)
y_pred_reg = regression_model.predict(X_test)

comparison = pd.DataFrame(
    {
        "actual_score": y_test_reg.values[:10],
        "predicted_score": y_pred_reg[:10].round(1),
    }
)
comparison

### Regression Metrics

- **MAE**: Mean Absolute Error. Average absolute prediction error.
- **RMSE**: Root Mean Squared Error. Penalizes large errors more strongly.
- **R²**: How much variation in the target is explained by the model. Higher is better.

In [ ]:
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 score:", round(r2, 3))

In [ ]:
# Plot actual vs predicted values.
plt.figure(figsize=(6, 6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.7)
plt.plot([0, 100], [0, 100], linestyle="--")
plt.title("Actual vs Predicted Final Score")
plt.xlabel("Actual Score")
plt.ylabel("Predicted Score")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.show()

## 14. Cross-Validation

A single train-test split can be lucky or unlucky. Cross-validation tests the model on multiple splits and gives a more stable estimate.

In [ ]:
cv_scores = cross_val_score(
    classification_model,
    X,
    y_classification,
    cv=5,
    scoring="accuracy",
)

print("Cross-validation scores:", cv_scores.round(3))
print("Mean accuracy:", round(cv_scores.mean(), 3))

## 15. Hyperparameter Tuning

Hyperparameters are model settings chosen before training. Grid search tries different settings and finds the best combination.

In [ ]:
forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=RANDOM_SEED)),
    ]
)

param_grid = {
    "model__n_estimators": [50, 100],
    "model__max_depth": [3, 5, None],
}

grid_search = GridSearchCV(
    forest_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train_cls)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", round(grid_search.best_score_, 3))

best_model = grid_search.best_estimator_
best_predictions = best_model.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test_cls, best_predictions), 3))

## 16. Clustering

Clustering is unsupervised learning. It finds groups without using a target label.

Here we group students based on numeric behavior.

In [ ]:
cluster_features = clean_data[["study_hours", "attendance_rate", "previous_score", "sleep_hours"]]

cluster_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=10)),
    ]
)

clusters = cluster_pipeline.fit_predict(cluster_features)
clean_data["student_cluster"] = clusters

clean_data.groupby("student_cluster")[["study_hours", "attendance_rate", "previous_score", "sleep_hours", "final_score"]].mean().round(1)

## 17. PCA: Dimensionality Reduction

PCA reduces many numeric columns into fewer dimensions while preserving as much variation as possible.

This is useful for visualization and sometimes for modeling.

In [ ]:
pca_input = SimpleImputer(strategy="median").fit_transform(cluster_features)
pca_input_scaled = StandardScaler().fit_transform(pca_input)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
pca_result = pca.fit_transform(pca_input_scaled)

plt.figure(figsize=(8, 5))
plt.scatter(pca_result[:, 0], pca_result[:, 1], c=clean_data["student_cluster"], alpha=0.7)
plt.title("Student Clusters Shown with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_.round(3))

## 18. Make Predictions for New Students

After training, we can use the model on new examples.

In [10]:
new_students = pd.DataFrame(
    {
        "study_hours": [8.0, 2.0, 5.5],
        "attendance_rate": [92, 55, 75],
        "previous_score": [78, 42, 61],
        "sleep_hours": [7.5, 5.0, 6.8],
        "internet_access": ["Yes", "No", "Yes"],
        "parent_education": ["Bachelor", "High School", "Master"],
        "extra_classes": ["Yes", "No", "No"],
    }
)

new_students["study_attendance_index"] = new_students["study_hours"] * (new_students["attendance_rate"] / 100)

new_pass_predictions = best_model.predict(new_students)
new_score_predictions = regression_model.predict(new_students)

new_students["predicted_passed"] = new_pass_predictions
new_students["predicted_final_score"] = new_score_predictions.round(1)

new_students

NameError: name 'best_model' is not defined

## 19. Save and Load a Model

A trained model can be saved and reused later. This is important for real applications.

In [ ]:
joblib.dump(best_model, "student_pass_prediction_model.joblib")
print("Saved: student_pass_prediction_model.joblib")

loaded_model = joblib.load("student_pass_prediction_model.joblib")
loaded_model.predict(new_students[feature_columns])

## 20. Communicating Results

A good data scientist explains results clearly.

Example summary:

- Study hours, attendance, and previous scores are strongly related to final performance.
- The classification model predicts pass/fail with measurable accuracy.
- The regression model estimates final score and should be evaluated using MAE, RMSE, and R².
- Clustering can group students into behavior profiles, such as high-attendance students or low-study students.

Important caution: A model can support decision-making, but it should not replace teacher judgment. Real educational data may include bias, missing context, and privacy concerns.

## 21. Student Exercises

Try these activities:

1. Change the passing score from 70 to 60 and retrain the model.
2. Add a new feature called `screen_time_hours` and see how it affects performance.
3. Compare Logistic Regression with Random Forest.
4. Try different KMeans cluster counts such as 2, 4, and 5.
5. Create a bar chart showing average final score by parent education.
6. Remove one feature at a time and observe whether model performance improves or worsens.
7. Explain the difference between classification and regression using this dataset.
8. Write a short report: what would you recommend to help students improve?

In [ ]:
# Exercise starter: average final score by parent education
avg_scores = clean_data.groupby("parent_education")["final_score"].mean().sort_values()

plt.figure(figsize=(8, 5))
avg_scores.plot(kind="bar", edgecolor="black")
plt.title("Average Final Score by Parent Education")
plt.xlabel("Parent Education")
plt.ylabel("Average Final Score")
plt.xticks(rotation=30)
plt.show()

## 22. Key Takeaways

- Data science starts with a clear question.
- Clean data matters as much as model choice.
- EDA helps us understand patterns and problems.
- Pipelines make preprocessing and modeling safer.
- Classification predicts categories.
- Regression predicts numbers.
- Evaluation metrics tell us whether a model is useful.
- Data science results should be communicated with context and responsibility.